# Module 2: Epidemic Modeling

## Team Members:
Sara Elster, Dani Folks

## Project Title:
*(Fill in)*

## Project Goal:
This project seeks to... *(what is the purpose of your project -- i.e., describe the question that you seek to answer by analyzing data.)*

## 1. Data and disease background
You can fill out this section throughout the module as you uncover more information about the mystery disease.

By the end of the module (when submitting), you should have some information about each of the following points:
* Prevalence & incidence in the UVA population
* Economic burden (you can generalize from respiratory viruses)
* Symptoms
* Biological mechanisms (anatomy, organ physiology, cell & molecular physiology - you can generalize from viral biology)


## 2. Data Analysis
This section should be filled out sequentially as a full report of the work you've done over this module. You can copy and paste code from any main.py file here, and run it to produce plots. Once you gain more information throughout the module, you do not need to go back and "fix" earlier results. In other words, if your early predictions are found to be wrong when gaining new data, do not go back and rewrite them.

### 2a. Methods

*IN A SUMMARY, DESCRIBE THE METHODS YOU USED TO ANALYZE AND MODEL THE DATA.*


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #1</b> 

</div>



### 2b. Plot the data & estimate initial growth rate (R0) from early data (through day 45)
This section should come from your python code after Data Release #1.

First, we imported the libraries necessary for computation and loaded the data from release 1.

In [ ]:
# Importing necessary libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

# Load the data
data = pd.read_csv(r'Data/mystery_virus_daily_active_counts_RELEASE#1.csv', parse_dates=['date'], header=0, index_col=None)

: 

Next, we used the Scipy curve_fit function to estimate R0 from the data.

In [ ]:
# We have day number, date, and active cases. We can use the day number and active cases to fit an exponential growth curve to estimate R0.
# Let's define the exponential growth function
def exponential_growth(t, r):
    return np.exp(r * t)

# Fit the exponential growth model to the data. 
# We'll use a handy function from scipy called CURVE_FIT that allows us to fit any given function to our data. 
# We will fit the exponential growth function to the active cases data. HINT: Look up the documentation for curve_fit to see how to use it.
x_data = data['day'].values.astype(float)
y_data = data['active reported daily cases'].values.astype(float)
popt, pcov = curve_fit(exponential_growth, x_data, y_data)

r_fit = popt[0]

# Approximate R0 using this fit
D = 6 # It seems from the data that there is a 6 day infectious period

r0 = np.exp(r_fit * D)

print("Estimated growth rate R0: ", r0)

Finally, we added the fitted exponential curve to the actual virus data to see whether it was a good estimate.

In [ ]:
# Add the fit as a line on top of your scatterplot.
# Generate fitted curve
y_fitted = exponential_growth(x_data, r_fit)

# Plot fitted curve and actual data
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label = "Actual Data")
plt.plot(x_data, y_fitted, color = "red", label = "Estimated Curve")
plt.xlabel('Day')
plt.ylabel('Active Cases')
plt.title('Exponential Growth Model Fit to Virus Data')
plt.xticks(rotation=45)
plt.tight_layout()
plt.legend()
plt.show() 

##### Preliminary Analysis/Conclusions:

**Analysis of Data Release 1:**
1. **What do you notice about initial infections?**

The initial infections seem to be very low, with only a few cases reported in the early days. This could indicate that the virus was not spreading widely at the beginning, or that there was limited testing and reporting.

2. **How could we measure how quickly its spreading?**

We could measure how quickly the virus is spreading by calculating the growth rate of active cases over time. This can be done by taking the difference in active cases between consecutive days and dividing it by the number of active cases on the previous day to get a percentage growth rate. 

3. **What information about the virus would be helpful in determining the shape of the outbreak curve?**

Information about the virus's transmission rate, incubation period, and recovery time would be helpful in determining the shape of the outbreak curve.


**From viruses.html:**
Mumps and chickenpox both have R0 values of 10. Mumps is a contagious disease caused by a virus that affects the salivary glands. It is spread through saliva and respiratory droplets (coughing/sneezing), and causes painful swelling of the salivary glands. Chickenpox is caused by the varicella-zoster virus and is spread through coughing, sneezing, and contact with chickenpox blister fluid. Symptoms include fever, fatigue, itchy rashes that turn into fluid filled blisters that scab over. Both diseases are highly contagious due to their easy transmission, explaining the high R0 value.

**How accurate do you think your R0 value is?**

Generally, we believe our R0 estimate is accurate. However, we are unsure of how to calculate D, so we guessed from observing the data that the infectious period was 6 days.


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #2</b> 

</div>



### 2c. Use Euler's method to solve the SEIR model.
This section should come from your python code after Data Release #2.

First, we imported the necessary libraries.

In [ ]:
#%% SEIR model fitting with Euler's method
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

Next, we pulled the dataset from Data Release 2 and organized the days and cases into their own data sets.

In [ ]:
# Helper: safely find data columns

data = pd.read_csv(r'Data/mystery_virus_daily_active_counts_RELEASE#2.csv', parse_dates=['date'], header=0, index_col=None)

possible_day_cols = ['day', 'day_number', 'Day', 'Day Number']
possible_case_cols = ['active_cases', 'active case', 'active', 'cases', 'Active Cases']

day_col = None
case_col = None

for col in data.columns:
    if col in possible_day_cols:
        day_col = col
    if col in possible_case_cols:
        case_col = col

# fallback: try lowercase matching
if day_col is None:
    for col in data.columns:
        if 'day' in col.lower():
            day_col = col
            break

if case_col is None:
    for col in data.columns:
        c = col.lower()
        if 'active' in c or 'case' in c:
            case_col = col
            break

if day_col is None or case_col is None:
    raise ValueError(f"Could not find required columns. Columns found: {list(data.columns)}")

timepoints = data[day_col].to_numpy()
infected_data = data[case_col].to_numpy()


Finally, we used Euler's method to approximate S, E, I, and R parameters of SEIR model.

In [ ]:
# Initial conditions
N = 10000           # total population
I0 = infected_data[0]
E0 = I0             # reasonable first guess: exposed starts similar to infected
R0_init = 0         # initial recovered is 0
S0 = N - E0 - I0 - R0_init

# Euler method for SEIR
def euler_seir(timepoints, N, S0, E0, I0, R0, beta, sigma, gamma):
    dt = timepoints[1] - timepoints[0]

    S = np.zeros(len(timepoints))
    E = np.zeros(len(timepoints))
    I = np.zeros(len(timepoints))
    R = np.zeros(len(timepoints))

    S[0] = S0
    E[0] = E0
    I[0] = I0
    R[0] = R0

    for i in range(len(timepoints) - 1):
        dSdt = -beta * S[i] * I[i] / N
        dEdt = beta * S[i] * I[i] / N - sigma * E[i]
        dIdt = sigma * E[i] - gamma * I[i]
        dRdt = gamma * I[i]

        S[i + 1] = S[i] + dSdt * dt
        E[i + 1] = E[i] + dEdt * dt
        I[i + 1] = I[i] + dIdt * dt
        R[i + 1] = R[i] + dRdt * dt

    return S, E, I, R

### 2d. Fit the SEIR model to the data by changing beta, gamma, and sigma.
This section should come from your python code after Data Release #2.

First, we used SSE to determine the optimal values for beta, sigma, and gamma, and the corresponding SSE.

In [ ]:
# SSE calculation
def calculate_sse(model_I, data_I):
    return np.sum((model_I - data_I) ** 2)

# Parameter search
def fit_seir_parameters(timepoints, N, S0, E0, I0, R0, infected_data):
    # These are reasonable starter ranges for a daily-timescale outbreak
    beta_values = np.linspace(0.1, 1.2, 30)
    sigma_values = np.linspace(0.05, 0.6, 25)
    gamma_values = np.linspace(0.05, 0.6, 25)

    best_sse = np.inf
    best_beta = None
    best_sigma = None
    best_gamma = None

    sse_results = []

    for b in beta_values:
        for s in sigma_values:
            for g in gamma_values:
                S, E, I, R = euler_seir(timepoints, N, S0, E0, I0, R0, b, s, g)
                sse = calculate_sse(I, infected_data)
                sse_results.append((b, s, g, sse))

                if sse < best_sse:
                    best_sse = sse
                    best_beta = b
                    best_sigma = s
                    best_gamma = g

    return best_beta, best_sigma, best_gamma, best_sse, sse_results

### 2e. Plot the model-predicted infections over time compared to the data.
This section should come from your python code after Data Release #2.

Finally, we ran the calculations and plotted the corresponding best fit model with the actual data.

In [ ]:
# Fit the model
best_beta, best_sigma, best_gamma, best_sse, sse_results = fit_seir_parameters(
    timepoints, N, S0, E0, I0, R0_init, infected_data
)

print("Best beta =", best_beta)
print("Best sigma =", best_sigma)
print("Best gamma =", best_gamma)
print("Best SSE =", best_sse)

# Run best-fit model on observed time window
S_fit, E_fit, I_fit, R_fit = euler_seir(
    timepoints, N, S0, E0, I0, R0_init,
    best_beta, best_sigma, best_gamma
)

# Plot model fit vs data
plt.figure(figsize=(8, 5))
plt.scatter(timepoints, infected_data, label='Observed active cases')
plt.plot(timepoints, I_fit, label='Best-fit SEIR I(t)')
plt.xlabel('Day')
plt.ylabel('Population')
plt.title('SEIR Fit to Epidemic Data')
plt.legend()
plt.show()

### 2e. Predict the day and amount of active cases at the peak of the epidemic spread.
This section should come from your python code after Data Release #2.

To predict the peak and plot the predicted graph, we calculated future timepoints until day 200 using the approximated S, E, I, and R values. Then, we used these values to predict what the peak number of cases would be and the corresponding day.

In [ ]:
# Predict the future peak
future_days = 200
future_timepoints = np.arange(timepoints[0], timepoints[0] + future_days + 1, 1)

S_future, E_future, I_future, R_future = euler_seir(
    future_timepoints, N, S0, E0, I0, R0_init,
    best_beta, best_sigma, best_gamma
)

peak_index = np.argmax(I_future)
peak_day = future_timepoints[peak_index]
peak_value = I_future[peak_index]

print("Predicted peak infected population =", peak_value)
print("Predicted peak day =", peak_day)


Finally, we used the predicted data to plot the predicted graph of number of cases vs days as well as the peak number of cases and days.

In [ ]:
# Plot future prediction
plt.figure(figsize=(8, 5))
plt.plot(future_timepoints, I_future, label='Predicted I(t)')
plt.scatter(peak_day, peak_value, label=f'Peak: day {peak_day}, I = {peak_value:.2f}')
plt.xlabel('Day')
plt.ylabel('Population')
plt.title('SEIR Future Prediction')
plt.legend()
plt.show()

#### Calculating True % Relative Error:
Actual Peak (based on Data Release 3): 3294 cases, on Day 83
1. For peak # of cases: % error = 5.178%
2. For peak day: % error = 4.819%


<div style="
    border-left: 6px solid #fbc02d;
    background-color: #fff8e1;
    padding: 10px 15px;
    border-radius: 4px;
">
<b style="color:#f57f17;">ANALYSIS AFTER DATA RELEASE #3</b> 

</div>



### 2f. Plot the full dataset (Data Release #3) against your model.
This section should come from your python code after Data Release #3.


In [ ]:
# Loading data from data release 3
data_3 = pd.read_csv(r'Data\mystery_virus_daily_active_counts_RELEASE#3.csv', parse_dates=['date'], header=0, index_col=None)

x_data_actual = data_3['day'].values.astype(float)
y_data_actual = data_3['active reported daily cases'].values.astype(float)

# Plotting full dataset against SEIR model
plt.figure(figsize=(8, 5))
plt.plot(future_timepoints, I_future, label='Predicted I(t)')
plt.scatter(peak_day, peak_value, label=f'Peak: day {peak_day}, I = {peak_value:.2f}')
plt.scatter(x_data_actual, y_data_actual, label = "Actual Data")
plt.xlabel('Day')
plt.ylabel('Population')
plt.title('SEIR Future Prediction')
plt.legend()
plt.show()


### 2g. Intervention strategies for new outbreak at VT (70 days of infection)
This section should come from your python code after Data Release #3.



Our team decided to test all five given strategies in order to see which method performed the best.

In [ ]:
# Loading data from data release 3
data_3 = pd.read_csv(r'Data/mystery_virus_daily_active_counts_RELEASE#3.csv', parse_dates=['date'], header=0, index_col=None)
x_data_actual = data_3['day'].values.astype(float)
y_data_actual = data_3['active reported daily cases'].values.astype(float)

# Best values for beta, sigma, and gamma from Euler calculation
best_beta = 0.25172413793103443
best_sigma = 0.4854166666666666
best_gamma = 0.07291666666666667

# VT Infection Day 0-70
def euler_seir(timepoints, N, S0, E0, I0, R0, beta, sigma, gamma):
    dt = timepoints[1] - timepoints[0]

    S = np.zeros(len(timepoints))
    E = np.zeros(len(timepoints))
    I = np.zeros(len(timepoints))
    R = np.zeros(len(timepoints))

    S[0] = S0
    E[0] = E0
    I[0] = I0
    R[0] = R0

    for i in range(len(timepoints) - 1):
        dSdt = -beta * S[i] * I[i] / N
        dEdt = beta * S[i] * I[i] / N - sigma * E[i]
        dIdt = sigma * E[i] - gamma * I[i]
        dRdt = gamma * I[i]

        S[i + 1] = S[i] + dSdt * dt
        E[i + 1] = E[i] + dEdt * dt
        I[i + 1] = I[i] + dIdt * dt
        R[i + 1] = R[i] + dRdt * dt

    return S, E, I, R


S0 = 38857 # VT student population
E0 = 1
I0 = 1
R0 = 0
timepoints = range(1,71)
N = int(S0) + int(E0) + int(I0) + int(R0)

S_fit, E_fit, I_fit, R_fit = euler_seir(
    timepoints, N, S0, E0, I0, R0,
    best_beta, best_sigma, best_gamma
)

plt.figure(figsize=(8, 5))
plt.plot(timepoints, I_fit, label='Best-fit SEIR I(t)')
plt.xlabel('Day')
plt.ylabel('Population')
plt.title('SEIR Fit to Epidemic Data')
plt.legend()
plt.show()


# ===== ENSURE DAY 70 STATE + TIMEPOINTS EXIST =====

# full baseline
timepoints_full = np.arange(1, 121)
S_base, E_base, I_base, R_base = euler_seir(
    timepoints_full, N, S0, E0, I0, R0,
    best_beta, best_sigma, best_gamma
)

# day 70 state
day70_index = np.where(timepoints_full == 70)[0][0]
S70 = S_base[day70_index]
E70 = E_base[day70_index]
I70 = I_base[day70_index]
R70 = R_base[day70_index]

# post-70 timepoints
timepoints_post70 = np.arange(70, 121)

# post-70 baseline
S_post_base, E_post_base, I_post_base, R_post_base = euler_seir(
    timepoints_post70, N, S70, E70, I70, R70,
    best_beta, best_sigma, best_gamma
)

# interventions
S_mask, E_mask, I_mask, R_mask = euler_seir(
    timepoints_post70, N, S70, E70, I70, R70,
    best_beta, best_sigma, best_gamma
)

# Intervention 1: Masking mandate
# reduces transmission by 40%

beta_mask = 0.6 * best_beta

S_mask, E_mask, I_mask, R_mask = euler_seir(
    timepoints_post70, N, S70, E70, I70, R70,
    beta_mask, best_sigma, best_gamma
)

# -----------------------------
# Intervention 2: Vaccine campaign
# single event on day 70
# vaccinate 2000 students with 90% efficacy
# move 1800 from S to R
# -----------------------------
effective_vax_campaign = 2000 * 0.90

S70_vax_campaign = max(S70 - effective_vax_campaign, 0)
R70_vax_campaign = R70 + min(effective_vax_campaign, S70)

S_vax_campaign, E_vax_campaign, I_vax_campaign, R_vax_campaign = euler_seir(
    timepoints_post70, N, S70_vax_campaign, E70, I70, R70_vax_campaign,
    best_beta, best_sigma, best_gamma
)

# -----------------------------
# Intervention 3: Vaccine rollout
# vaccinate 1000 students on day 70, 80, 90 with 90% efficacy
# move 900 from S to R at each event
# -----------------------------
def euler_seir_vaccine_rollout(timepoints, N, S0, E0, I0, R0, beta, sigma, gamma, vax_days, vax_amount, efficacy):
    dt = timepoints[1] - timepoints[0]

    S = np.zeros(len(timepoints))
    E = np.zeros(len(timepoints))
    I = np.zeros(len(timepoints))
    R = np.zeros(len(timepoints))

    S[0] = S0
    E[0] = E0
    I[0] = I0
    R[0] = R0

    effective_vax = vax_amount * efficacy

    for i in range(len(timepoints) - 1):
        current_day = timepoints[i]

        # apply vaccination at the start of specified days
        if current_day in vax_days:
            moved = min(effective_vax, S[i])
            S[i] -= moved
            R[i] += moved

        dSdt = -beta * S[i] * I[i] / N
        dEdt = beta * S[i] * I[i] / N - sigma * E[i]
        dIdt = sigma * E[i] - gamma * I[i]
        dRdt = gamma * I[i]

        S[i + 1] = S[i] + dSdt * dt
        E[i + 1] = E[i] + dEdt * dt
        I[i + 1] = I[i] + dIdt * dt
        R[i + 1] = R[i] + dRdt * dt

    return S, E, I, R

S_vax_rollout, E_vax_rollout, I_vax_rollout, R_vax_rollout = euler_seir_vaccine_rollout(
    timepoints_post70, N, S70, E70, I70, R70,
    best_beta, best_sigma, best_gamma,
    vax_days=[70, 80, 90],
    vax_amount=1000,
    efficacy=0.90
)

# -----------------------------
# Intervention 4: Testing + quarantine
# reduces infectious period by 2 days
# infectious period = 1/gamma
# new gamma = 1/(old infectious period - 2)
# -----------------------------
infectious_period = 1 / best_gamma
new_infectious_period = infectious_period - 2

if new_infectious_period <= 0:
    raise ValueError("New infectious period is not valid. Check gamma.")

gamma_test = 1 / new_infectious_period

S_test, E_test, I_test, R_test = euler_seir(
    timepoints_post70, N, S70, E70, I70, R70,
    best_beta, best_sigma, gamma_test
)

# -----------------------------
# Intervention 5: 2-week school closure
# day 70-84: only 20% of normal contacts
# after closure: return to normal
# -----------------------------
def euler_seir_school_closure(timepoints, N, S0, E0, I0, R0,
                              beta_normal, sigma, gamma,
                              closure_start, closure_end, closure_contact_fraction):
    dt = timepoints[1] - timepoints[0]

    S = np.zeros(len(timepoints))
    E = np.zeros(len(timepoints))
    I = np.zeros(len(timepoints))
    R = np.zeros(len(timepoints))

    S[0] = S0
    E[0] = E0
    I[0] = I0
    R[0] = R0

    for i in range(len(timepoints) - 1):
        current_day = timepoints[i]

        if closure_start <= current_day < closure_end:
            beta_current = beta_normal * closure_contact_fraction
        else:
            beta_current = beta_normal

        dSdt = -beta_current * S[i] * I[i] / N
        dEdt = beta_current * S[i] * I[i] / N - sigma * E[i]
        dIdt = sigma * E[i] - gamma * I[i]
        dRdt = gamma * I[i]

        S[i + 1] = S[i] + dSdt * dt
        E[i + 1] = E[i] + dEdt * dt
        I[i + 1] = I[i] + dIdt * dt
        R[i + 1] = R[i] + dRdt * dt

    return S, E, I, R

S_close, E_close, I_close, R_close = euler_seir_school_closure(
    timepoints_post70, N, S70, E70, I70, R70,
    best_beta, best_sigma, best_gamma,
    closure_start=70,
    closure_end=84,
    closure_contact_fraction=0.20
)

# -----------------------------
# Metrics function
# Peak infections and total cases prevented from day 70-120
# cases = S(day70) - S(day120)
# -----------------------------
def intervention_metrics(name, S_int, I_int, S_base, I_base):
    peak_base = np.max(I_base)
    peak_int = np.max(I_int)

    total_cases_base = S_base[0] - S_base[-1]
    total_cases_int = S_int[0] - S_int[-1]

    peak_reduction = peak_base - peak_int
    cases_prevented = total_cases_base - total_cases_int

    print(f'--- {name} ---')
    print(f'Peak infections (baseline): {peak_base:.2f}')
    print(f'Peak infections (intervention): {peak_int:.2f}')
    print(f'Peak reduction: {peak_reduction:.2f}')
    print(f'Total cases day 70-120 (baseline): {total_cases_base:.2f}')
    print(f'Total cases day 70-120 (intervention): {total_cases_int:.2f}')
    print(f'Cases prevented: {cases_prevented:.2f}')
    print()

intervention_metrics('Masking mandate', S_mask, I_mask, S_post_base, I_post_base)
intervention_metrics('Vaccine campaign', S_vax_campaign, I_vax_campaign, S_post_base, I_post_base)
intervention_metrics('Vaccine rollout', S_vax_rollout, I_vax_rollout, S_post_base, I_post_base)
intervention_metrics('Testing + quarantine', S_test, I_test, S_post_base, I_post_base)
intervention_metrics('2-week school closure', S_close, I_close, S_post_base, I_post_base)

# -----------------------------
# Plot all interventions vs baseline
# -----------------------------
plt.figure(figsize=(10, 6))
plt.plot(timepoints_post70, I_post_base, label='Baseline')
plt.plot(timepoints_post70, I_mask, label='Masking mandate')
plt.plot(timepoints_post70, I_vax_campaign, label='Vaccine campaign')
plt.plot(timepoints_post70, I_vax_rollout, label='Vaccine rollout')
plt.plot(timepoints_post70, I_test, label='Testing + quarantine')
plt.plot(timepoints_post70, I_close, label='2-week school closure')

plt.xlabel('Day')
plt.ylabel('Active infections')
plt.title('VT Interventions Compared to Baseline (Days 70-120)')
plt.legend()
plt.show()



## Verify and validate your analysis: 

We verified our analysis by first checking that the SEIR code behaved logically at each step. We confirmed that the baseline simulation used the fitted values of beta, sigma, and gamma that each intervention began from the baseline day-70 compartment values rather than restarting from day 1, and that the model outputs were reasonable in shape and magnitude. We also compared the fitted baseline curve directly to the observed epidemic data across the full 120-day period to ensure the model captured the rise, peak, and decline of infections rather than only the early exponential phase. In addition, we checked that intervention effects matched expectations mathematically: masking lowered transmission by reducing beta, vaccination moved effectively protected individuals from 𝑆 to 𝑅, testing/quarantine increased gamma by shortening the infectious period, and school closure temporarily reduced contact rates.

We validated the analysis by comparing the model’s intervention trends to published evidence. Prior studies show that masking in school and university settings reduces transmission, vaccination lowers campus outbreak burden, and testing with isolation can substantially reduce spread when compliance is adequate. These findings support the direction of the model results, even though the exact numerical outcomes depend on simplifying assumptions such as homogeneous mixing and fixed parameter values.
https://pmc.ncbi.nlm.nih.gov/articles/PMC8437055/?

## Conclusions and Ethical Implications: 
The SEIR model results indicate that all interventions reduce infections compared to the no-intervention baseline, but their effectiveness varies in magnitude and practicality. Masking produced an immediate reduction in transmission, lowering peak infections significantly. Vaccination strategies, particularly the phased rollout, reduced both peak infections and total cases over time by decreasing the susceptible population. Testing and quarantine reduced the infectious period and helped suppress spread, while the two-week school closure produced a sharp temporary decrease in infections but allowed cases to rebound once normal contact resumed. Overall, a combination of masking and vaccination provided the most balanced and sustainable reduction in both peak infections and total cases.

From an ethical perspective, these interventions involve trade-offs between individual autonomy, public health, and social impact. Masking mandates and vaccination campaigns raise concerns about personal choice and compliance but are relatively low-cost and broadly beneficial to the community. Testing and quarantine rely heavily on individual responsibility and can disproportionately affect students who lack resources to isolate effectively. School closures, while effective in reducing transmission, carry significant academic, social, and mental health consequences and may disproportionately impact vulnerable students. Therefore, the most ethical approach is one that maximizes public health benefits while minimizing harm and inequity—favoring interventions like masking and vaccination that are effective, less disruptive, and more equitable across the student population.

## Limitations and Future Work: 
This analysis is subject to several limitations due to simplifying assumptions in the SEIR model. The model assumes a homogeneous, well-mixed population, meaning every student has an equal probability of interacting with every other student. In reality, contact patterns on a college campus are highly structured (e.g., dorms, classes, social groups), which can significantly affect transmission dynamics. Additionally, the model uses constant parameters for transmission, incubation, and recovery, whereas real-world behavior and policy changes over time can alter these values. For example, compliance with masking or testing may vary, and vaccination uptake is unlikely to be perfectly uniform. The model also assumes perfect implementation of interventions (e.g., exact numbers vaccinated, consistent reduction in contacts), which may not reflect real-world logistical challenges or human behavior.

Another limitation is that the model does not account for factors such as asymptomatic transmission, delays in testing and reporting, reinfection, or differences in disease severity. These factors could influence both the timing and magnitude of the epidemic peak. Additionally, the initial conditions and parameter estimates are derived from available data, which may include reporting noise or undercounting, affecting the accuracy of the fitted model.

Future work could improve this analysis by incorporating more realistic features, such as network-based or agent-based models that capture structured interactions within the student population. Time-varying parameters could be introduced to reflect changes in behavior or policy over the course of the outbreak. Including stochastic elements would allow the model to capture uncertainty and variability rather than producing a single deterministic outcome. Additionally, integrating real-world compliance rates and logistical constraints would make intervention simulations more realistic. Finally, validating the model against multiple datasets or published studies would strengthen confidence in the results and improve its applicability to real-world decision-making.